In [ ]:
"""
Excess Phase Decomposition for 3D Acoustic Directivity Patterns
================================================================
Given complex impulse responses H(f, Omega) measured on a sphere
(e.g. 256-mic array), this module decomposes the excess phase into:

  1. Common (direction-independent) component  -> monopole resonances
  2. Acoustic center shift                     -> dipole phase, l=1
  3. Spatially structured residual             -> SH decomposition l>=2
  4. Noise floor estimation

Usage
-----
See bottom of file for a worked example with synthetic data.
"""

import numpy as np
from scipy.signal import hilbert
from scipy.linalg import lstsq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Compatibility for scipy >= 1.15 which removed sph_harm
try:
    from scipy.special import sph_harm as _sph_harm_orig
    def _sph(m, l, phi_az, theta):
        return _sph_harm_orig(m, l, phi_az, theta)
except ImportError:
    from scipy.special import sph_harm_y
    def _sph(m, l, phi_az, theta):
        return sph_harm_y(l, m, theta, phi_az)


# ---------------------------------------------------------------------------
# 1.  CORE PHASE UTILITIES
# ---------------------------------------------------------------------------

def compute_minimum_phase(H):
    """
    Compute the minimum-phase equivalent of each transfer function.

    Parameters
    ----------
    H : ndarray, shape (n_freq, n_mics)
        Complex transfer functions (single-sided FFT, DC to Nyquist).

    Returns
    -------
    phi_min : ndarray, shape (n_freq, n_mics)
        Minimum-phase response in radians.
    """
    log_mag = np.log(np.abs(H) + 1e-12)
    # Hilbert transform along frequency axis:
    #   phi_min(f) = -imag{ hilbert( log|H(f)| ) }
    phi_min = -np.imag(hilbert(log_mag, axis=0))
    return phi_min


def compute_excess_phase(H):
    """
    Compute actual, minimum-phase, and excess phase.

    Returns
    -------
    phi_actual, phi_min, phi_excess : ndarray, shape (n_freq, n_mics)
    """
    phi_actual = np.unwrap(np.angle(H), axis=0)
    phi_min    = compute_minimum_phase(H)

    # Align DC constant
    offset  = phi_actual[0, :] - phi_min[0, :]
    phi_min = phi_min + offset[np.newaxis, :]

    phi_excess = phi_actual - phi_min
    return phi_actual, phi_min, phi_excess


# ---------------------------------------------------------------------------
# 2.  SPHERICAL HARMONICS BASIS
# ---------------------------------------------------------------------------

def real_sh_basis(theta, phi_az, l_max):
    """
    Build real spherical harmonics basis matrix Y, shape (n_dirs, n_sh).

    Parameters
    ----------
    theta  : colatitude in [0, pi]   (n_dirs,)
    phi_az : azimuth   in [0, 2pi]   (n_dirs,)
    l_max  : maximum SH order

    Returns
    -------
    Y      : ndarray (n_dirs, (l_max+1)**2)
    labels : list of (l, m) tuples
    """
    n_dirs = len(theta)
    n_sh   = (l_max + 1) ** 2
    Y      = np.zeros((n_dirs, n_sh))
    labels = []

    idx = 0
    for l in range(l_max + 1):
        for m in range(-l, l + 1):
            if m < 0:
                Ylm = np.sqrt(2) * (-1)**m * np.imag(_sph(abs(m), l, phi_az, theta))
            elif m == 0:
                Ylm = np.real(_sph(0, l, phi_az, theta))
            else:
                Ylm = np.sqrt(2) * (-1)**m * np.real(_sph(m, l, phi_az, theta))
            Y[:, idx] = Ylm
            labels.append((l, m))
            idx += 1

    return Y, labels


# ---------------------------------------------------------------------------
# 3.  DECOMPOSITION STEPS
# ---------------------------------------------------------------------------

def remove_common_phase(phi_excess):
    """
    Step 1: Direction-averaged (common) excess phase.

    Returns
    -------
    phi_common   : ndarray (n_freq,)        – mean over directions
    phi_residual : ndarray (n_freq, n_mics) – phi_excess - phi_common
    """
    phi_common   = np.mean(phi_excess, axis=1)
    phi_residual = phi_excess - phi_common[:, np.newaxis]
    return phi_common, phi_residual


def fit_acoustic_center(phi_residual, freqs, theta, phi_az, c=343.0):
    """
    Step 2: Fit direction-dependent linear-in-frequency phase (acoustic center).

    Model: phi_residual(f, Omega) ≈ -2*pi*f * (d(f)·Omega_hat) / c

    Returns
    -------
    tau_excess    : ndarray (n_freq, n_dirs) – group-delay field (seconds)
    d_vec         : ndarray (n_freq, 3)      – acoustic center (meters)
    phi_dipole    : ndarray (n_freq, n_dirs) – fitted dipole phase
    phi_residual2 : ndarray (n_freq, n_dirs) – residual after dipole removal
    """
    n_freq, n_dirs = phi_residual.shape

    df         = freqs[1] - freqs[0]
    dphi       = np.gradient(phi_residual, df, axis=0)
    tau_excess = -dphi / (2 * np.pi)    # seconds

    # Direction cosines
    sin_t = np.sin(theta)
    Omega = np.column_stack([sin_t * np.cos(phi_az),
                             sin_t * np.sin(phi_az),
                             np.cos(theta)])           # (n_dirs, 3)

    d_vec      = np.zeros((n_freq, 3))
    phi_dipole = np.zeros_like(phi_residual)

    for fi, f in enumerate(freqs):
        if f < 1e-6:
            continue
        d, _, _, _ = lstsq(Omega, tau_excess[fi])
        d_vec[fi]  = d * c
        phi_dipole[fi] = -2 * np.pi * f * (Omega @ d) / c

    phi_residual2 = phi_residual - phi_dipole
    return tau_excess, d_vec, phi_dipole, phi_residual2


def sh_decomposition(phi_residual2, theta, phi_az, l_max=6):
    """
    Step 3: Decompose remaining excess phase into spherical harmonics.

    Returns
    -------
    coeffs         : ndarray (n_freq, n_sh)
    Y              : ndarray (n_dirs, n_sh)
    labels         : list of (l, m)
    power_by_order : ndarray (n_freq, l_max+1) – energy per SH order
    """
    Y, labels = real_sh_basis(theta, phi_az, l_max)
    n_freq = phi_residual2.shape[0]

    coeffs, _, _, _ = lstsq(Y, phi_residual2.T)
    coeffs = coeffs.T   # (n_freq, n_sh)

    power_by_order = np.zeros((n_freq, l_max + 1))
    for i, (l, m) in enumerate(labels):
        power_by_order[:, l] += coeffs[:, i] ** 2

    return coeffs, Y, labels, power_by_order


def excess_phase_norm(phi_excess):
    """RMS norm over frequency for each direction."""
    return np.sqrt(np.mean(phi_excess ** 2, axis=0))

In [3]:
# ---------------------------------------------------------------------------
# 4.  FULL PIPELINE
# ---------------------------------------------------------------------------

def decompose_directivity(H, freqs, theta, phi_az, c=343.0, l_max=6):
    """
    Full excess phase decomposition pipeline.

    Parameters
    ----------
    H      : complex ndarray (n_freq, n_dirs)
    freqs  : ndarray (n_freq,)  in Hz
    theta  : colatitude (n_dirs,) in radians
    phi_az : azimuth    (n_dirs,) in radians
    c      : speed of sound (m/s)
    l_max  : max SH order for residual decomposition

    Returns
    -------
    dict with all intermediate and final quantities
    """
    phi_actual, phi_min, phi_excess = compute_excess_phase(H)

    phi_common, phi_res1 = remove_common_phase(phi_excess)

    tau_excess, d_vec, phi_dipole, phi_res2 = fit_acoustic_center(
        phi_res1, freqs, theta, phi_az, c=c)

    coeffs, Y, labels, power_by_order = sh_decomposition(
        phi_res2, theta, phi_az, l_max=l_max)

    # Noise estimate: residual after removing all fitted SH components
    high_l_mask = np.array([l for l, m in labels]) >= l_max
    phi_noise   = phi_res2 - (Y[:, ~high_l_mask] @ coeffs[:, ~high_l_mask].T).T

    return dict(
        freqs=freqs, theta=theta, phi_az=phi_az,
        phi_actual=phi_actual, phi_min=phi_min, phi_excess=phi_excess,
        phi_common=phi_common,
        phi_res1=phi_res1,
        tau_excess=tau_excess,
        d_vec=d_vec,
        phi_dipole=phi_dipole,
        phi_res2=phi_res2,
        coeffs=coeffs, Y=Y, labels=labels,
        power_by_order=power_by_order,
        phi_noise=phi_noise,
    )


In [4]:
# ---------------------------------------------------------------------------
# 5.  PLOTTING
# ---------------------------------------------------------------------------

def plot_decomposition(res, violin_id="", f_range=None, outfile=None):
    """
    Summary figure with 5 panels:
      A) Excess phase norm histogram (total vs components)
      B) Common phase group delay (monopole resonances)
      C) Acoustic center displacement |d(f)|
      D) SH power by order vs frequency
      E) Residual noise map on sphere (RMS, Mollweide projection)
    """
    freqs = res['freqs']
    mask  = ((freqs >= f_range[0]) & (freqs <= f_range[1])
             if f_range else np.ones(len(freqs), bool))
    f = freqs[mask]

    fig = plt.figure(figsize=(14, 12))
    fig.suptitle(f"Excess Phase Decomposition — Violin {violin_id}", fontsize=14)
    gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    # A: Histogram of norms per component
    ax = fig.add_subplot(gs[0, :])
    norm_total    = excess_phase_norm(res['phi_excess'][mask])
    norm_dipole   = excess_phase_norm(res['phi_dipole'][mask])
    norm_residual = excess_phase_norm(res['phi_res2'][mask])
    bins = np.linspace(0, norm_total.max() * 1.1, 40)
    ax.hist(norm_total,    bins=bins, alpha=0.5, label='Total excess phase',       color='steelblue')
    ax.hist(norm_dipole,   bins=bins, alpha=0.6, label='Dipole (acoustic center)', color='orange')
    ax.hist(norm_residual, bins=bins, alpha=0.6, label='Structured residual (l≥2)',color='green')
    ax.set_xlabel("Excess Phase Norm (rad)")
    ax.set_ylabel("Count")
    ax.set_title("A — Norm distributions per component")
    ax.legend(fontsize=9)

    # B: Common group delay
    ax = fig.add_subplot(gs[1, 0])
    df = f[1] - f[0]
    gd = -np.gradient(res['phi_common'][mask], df) / (2 * np.pi) * 1e3  # ms
    ax.plot(f, gd, 'k')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Group delay (ms)")
    ax.set_title("B — Common group delay (monopole resonances)")

    # C: Acoustic center displacement
    ax = fig.add_subplot(gs[1, 1])
    d_norm = np.linalg.norm(res['d_vec'][mask], axis=1) * 100  # cm
    ax.plot(f, d_norm, 'r')
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("|d(f)| (cm)")
    ax.set_title("C — Acoustic center displacement")

    # D: SH power by order
    ax = fig.add_subplot(gs[2, 0])
    po      = res['power_by_order'][mask]
    po_norm = po / (po.sum(axis=1, keepdims=True) + 1e-12)
    l_max_p = po.shape[1]
    cmap    = plt.cm.hsv(np.linspace(0, 1, l_max_p))
    for l in range(l_max_p):
        ax.plot(f, po_norm[:, l], color=cmap[l], label=f"l={l}")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Relative SH energy")
    ax.set_title("D — SH power by order (residual)")
    ax.legend(fontsize=7, ncol=2)

    # E: Residual noise on sphere (Mollweide)
    ax  = fig.add_subplot(gs[2, 1], projection='mollweide')
    rms = excess_phase_norm(res['phi_noise'][mask])
    lon = res['phi_az'].copy()
    lon[lon > np.pi] -= 2 * np.pi
    lat = np.pi / 2 - res['theta']
    sc  = ax.scatter(lon, lat, c=rms, cmap='hot_r', s=10)
    plt.colorbar(sc, ax=ax, label='RMS (rad)', shrink=0.7)
    ax.set_title("E — Noise/residual RMS on sphere")

    if outfile:
        plt.savefig(outfile, bbox_inches='tight')
        print(f"Saved: {outfile}")
    else:
        plt.savefig("excess_phase_decomposition.pdf", bbox_inches='tight')
    plt.close(fig)
    return fig


In [5]:
C = 343.0  # Speed of sound in m/s
H = np.load(f'./../results/Optimized_Cmn_and_Diag.npz')['Ref_Diag']
freqs = np.load(f'./../results/Optimized_Cmn_and_Diag.npz')['kvect']*C/(2*np.pi)
theta = np.load(f'./../results/Optimized_Cmn_and_Diag.npz')['angles_look'][:,0]
phi_az = np.load(f'./../results/Optimized_Cmn_and_Diag.npz')['angles_look'][:,1]

In [6]:
f_range = (100, 10000)  # Hz
for nv in range(6):
    res = decompose_directivity(H[nv,:,:].T, freqs, theta, phi_az, c=C, l_max=7)

    # Report
    mask = (freqs >= f_range[0]) & (freqs <= f_range[1])
    print("=== Excess Phase Decomposition ===")
    print(f"Mean excess phase norm (total):    {excess_phase_norm(res['phi_excess'][mask]).mean():.3f} rad")
    print(f"Mean norm (dipole component):      {excess_phase_norm(res['phi_dipole'][mask]).mean():.3f} rad")
    print(f"Mean norm (structured residual):   {excess_phase_norm(res['phi_res2'][mask]).mean():.3f} rad")
    d_cm = np.linalg.norm(res['d_vec'][mask], axis=1).mean() * 100
    print(f"Mean acoustic center displacement: {d_cm:.1f} cm")
    plot_decomposition(res, violin_id = f"#{NumViolon[nv]}", f_range = f_range, outfile=f"excess_phase_decomposition_{nv}.pdf")



C:\Users\froll\AppData\Local\Temp\ipykernel_30712\3311802718.py:29: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  return _sph_harm_orig(m, l, phi_az, theta)


=== Excess Phase Decomposition ===
Mean excess phase norm (total):    102.971 rad
Mean norm (dipole component):      0.059 rad
Mean norm (structured residual):   12.956 rad
Mean acoustic center displacement: 38.0 cm
Saved: excess_phase_decomposition_0.pdf


C:\Users\froll\AppData\Local\Temp\ipykernel_30712\3311802718.py:29: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  return _sph_harm_orig(m, l, phi_az, theta)


=== Excess Phase Decomposition ===
Mean excess phase norm (total):    119.219 rad
Mean norm (dipole component):      0.076 rad
Mean norm (structured residual):   14.428 rad
Mean acoustic center displacement: 49.5 cm
Saved: excess_phase_decomposition_1.pdf


C:\Users\froll\AppData\Local\Temp\ipykernel_30712\3311802718.py:29: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  return _sph_harm_orig(m, l, phi_az, theta)


...

=== Excess Phase Decomposition ===
Mean excess phase norm (total):    114.452 rad
Mean norm (dipole component):      0.072 rad
Mean norm (structured residual):   14.448 rad
Mean acoustic center displacement: 45.1 cm
Saved: excess_phase_decomposition_3.pdf


C:\Users\froll\AppData\Local\Temp\ipykernel_30712\3311802718.py:29: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  return _sph_harm_orig(m, l, phi_az, theta)


=== Excess Phase Decomposition ===
Mean excess phase norm (total):    114.414 rad
Mean norm (dipole component):      0.070 rad
Mean norm (structured residual):   14.695 rad
Mean acoustic center displacement: 45.6 cm
Saved: excess_phase_decomposition_4.pdf


C:\Users\froll\AppData\Local\Temp\ipykernel_30712\3311802718.py:29: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  return _sph_harm_orig(m, l, phi_az, theta)


=== Excess Phase Decomposition ===
Mean excess phase norm (total):    113.007 rad
Mean norm (dipole component):      0.063 rad
Mean norm (structured residual):   16.750 rad
Mean acoustic center displacement: 46.7 cm
Saved: excess_phase_decomposition_5.pdf


In [14]:
"""
Acoustic Center 3D Trajectory Extraction
=========================================
Extracts and visualizes the frequency-dependent acoustic center d(f)
from a 3D complex directivity pattern H(f, Omega).

Key improvement over group-delay approach:
  - Direct phase fit at each frequency (no differentiation → less noise)
  - Savitzky-Golay smoothing of d_vec(f) components
  - 3D trajectory plot colored by frequency
  - Octave-band summary statistics

Can be used standalone or imported into excess_phase_decomposition.py
"""

import numpy as np
from scipy.linalg import lstsq
from scipy.signal import savgol_filter, hilbert
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Scipy sph_harm compatibility
try:
    from scipy.special import sph_harm as _sph_harm_orig
    def _sph(m, l, phi_az, theta):
        return _sph_harm_orig(m, l, phi_az, theta)
except ImportError:
    from scipy.special import sph_harm_y
    def _sph(m, l, phi_az, theta):
        return sph_harm_y(l, m, theta, phi_az)


# ---------------------------------------------------------------------------
# 1.  PHASE PREPARATION
# ---------------------------------------------------------------------------

def compute_minimum_phase(H):
    log_mag = np.log(np.abs(H) + 1e-12)
    return -np.imag(hilbert(log_mag, axis=0))


def compute_excess_phase(H):
    phi_actual = np.unwrap(np.angle(H), axis=0)
    phi_min    = compute_minimum_phase(H)
    offset     = phi_actual[0, :] - phi_min[0, :]
    phi_min   += offset[np.newaxis, :]
    return phi_actual - phi_min


def remove_common_phase(phi_excess):
    phi_common = np.mean(phi_excess, axis=1)
    return phi_common, phi_excess - phi_common[:, np.newaxis]


# ---------------------------------------------------------------------------
# 2.  DIRECT PHASE FIT (more stable than group-delay approach)
# ---------------------------------------------------------------------------

def fit_acoustic_center_phase(phi_res1, freqs, theta, phi_az, c=343.0,
                              min_freq=50.0):
    """
    Fit acoustic center by direct phase regression at each frequency.

    Model:  phi_res1(f, Omega) = -2*pi*f/c * (d . Omega_hat)
    Solve:  d(f) = -c/(2*pi*f) * pinv(Omega) @ phi_res1(f, :)

    Parameters
    ----------
    phi_res1 : ndarray (n_freq, n_dirs)  – excess phase after common removal
    freqs    : ndarray (n_freq,)  Hz
    theta    : colatitude (n_dirs,)  rad
    phi_az   : azimuth    (n_dirs,)  rad
    c        : speed of sound (m/s)
    min_freq : minimum frequency for reliable fit (Hz)

    Returns
    -------
    d_vec      : ndarray (n_freq, 3)  acoustic center position (meters)
    fit_error  : ndarray (n_freq,)    RMS fit residual (rad) per frequency
    phi_dipole : ndarray (n_freq, n_dirs) reconstructed dipole phase
    """
    n_freq, n_dirs = phi_res1.shape

    sin_t = np.sin(theta)
    Omega = np.column_stack([sin_t * np.cos(phi_az),
                             sin_t * np.sin(phi_az),
                             np.cos(theta)])              # (n_dirs, 3)

    d_vec      = np.zeros((n_freq, 3))
    fit_error  = np.zeros(n_freq)
    phi_dipole = np.zeros_like(phi_res1)

    # Precompute pseudo-inverse of Omega (constant for all frequencies)
    Omega_pinv = np.linalg.pinv(Omega)                   # (3, n_dirs)

    for fi, f in enumerate(freqs):
        if f < min_freq:
            continue
        # Direct solve: phi = -2pi*f/c * Omega @ d  =>  d = -c/(2pi*f) * Omega_pinv @ phi
        d = -c / (2 * np.pi * f) * (Omega_pinv @ phi_res1[fi])
        d_vec[fi]      = d
        phi_dipole[fi] = -2 * np.pi * f / c * (Omega @ d)
        residual       = phi_res1[fi] - phi_dipole[fi]
        fit_error[fi]  = np.sqrt(np.mean(residual ** 2))

    return d_vec, fit_error, phi_dipole


def smooth_trajectory(d_vec, freqs, window_oct=0.5, polyorder=3,
                      min_freq=50.0):
    """
    Smooth d_vec(f) with a Savitzky-Golay filter on a log-frequency scale.

    window_oct : smoothing window in octaves
    """
    d_smooth = d_vec.copy()
    mask = freqs >= min_freq

    if mask.sum() < polyorder + 2:
        return d_smooth

    # Convert octave window to number of frequency samples
    # Uniform spacing in Hz: approximate octave window at geometric mean
    f_mid     = np.exp(np.mean(np.log(freqs[mask])))
    df        = freqs[1] - freqs[0]
    win_hz    = f_mid * (2 ** (window_oct / 2) - 2 ** (-window_oct / 2))
    win_samp  = max(int(win_hz / df) | 1, polyorder + 2)
    if win_samp % 2 == 0:
        win_samp += 1

    for axis in range(3):
        d_smooth[mask, axis] = savgol_filter(
            d_vec[mask, axis], win_samp, polyorder)

    return d_smooth


# ---------------------------------------------------------------------------
# 3.  OCTAVE-BAND SUMMARY
# ---------------------------------------------------------------------------

OCTAVE_CENTERS = np.array([125, 250, 500, 1000, 2000, 4000, 8000])

def octave_band_centers(d_smooth, freqs, centers=OCTAVE_CENTERS):
    """
    Return the acoustic center position at each octave band center.

    Returns
    -------
    d_bands : ndarray (n_bands, 3)  – positions in cm
    valid   : bool array (n_bands,)
    """
    d_bands = np.zeros((len(centers), 3))
    valid   = np.zeros(len(centers), dtype=bool)
    for i, fc in enumerate(centers):
        f_lo, f_hi = fc / np.sqrt(2), fc * np.sqrt(2)
        idx = np.where((freqs >= f_lo) & (freqs <= f_hi))[0]
        if len(idx):
            d_bands[i] = d_smooth[idx].mean(axis=0) * 100  # cm
            valid[i]   = True
    return d_bands, valid


# ---------------------------------------------------------------------------
# 4.  VISUALIZATION
# ---------------------------------------------------------------------------

def plot_trajectory(d_raw, d_smooth, freqs, fit_error,
                    f_range=(200, 8000), violin_id="",
                    outfile=None):
    """
    4-panel figure:
      A) 3D trajectory colored by frequency
      B) X, Y, Z components vs frequency
      C) |d(f)| magnitude
      D) Fit error (RMS residual) — quality indicator
    """
    mask = (freqs >= f_range[0]) & (freqs <= f_range[1])
    f    = freqs[mask]
    dr   = d_raw[mask]    * 100   # cm
    ds   = d_smooth[mask] * 100   # cm
    err  = fit_error[mask]

    fig  = plt.figure(figsize=(14, 11))
    fig.suptitle(f"Acoustic Center 3D Trajectory — Violin {violin_id}",
                 fontsize=14)
    gs   = GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

    # --- A: 3D trajectory ---
    ax3d = fig.add_subplot(gs[0, 0], projection='3d')
    norm  = mcolors.LogNorm(vmin=f.min(), vmax=f.max())
    cmap  = cm.plasma
    colors = cmap(norm(f))

    # Draw as colored line segments
    points  = ds[:, np.newaxis, :]           # (N, 1, 3)
    segs    = np.concatenate([points[:-1], points[1:]], axis=1)
    lc      = Line3DCollection(segs, colors=colors[:-1], linewidth=2)
    ax3d.add_collection3d(lc)

    # Octave-band markers
    d_bands, valid = octave_band_centers(d_smooth, freqs)
    for i, (fc, ok) in enumerate(zip(OCTAVE_CENTERS, valid)):
        if ok and f_range[0] <= fc <= f_range[1]:
            x, y, z = d_bands[i]
            c_norm  = norm(fc)
            ax3d.scatter(x, y, z, s=80, color=cmap(c_norm),
                         edgecolors='k', linewidths=0.8, zorder=5)
            ax3d.text(x, y, z + 0.3, f"{fc}Hz", fontsize=6, ha='center')

    # Reference origin
    ax3d.scatter(0, 0, 0, s=100, color='red', marker='+', zorder=10)

    #lim = max(np.abs(ds).max(), 1.0) * 1.2
    lim = 30
    ax3d.set_xlim(-lim, lim); ax3d.set_ylim(-lim, lim); ax3d.set_zlim(-lim, lim)
    ax3d.set_xlabel("X (cm)"); ax3d.set_ylabel("Y (cm)"); ax3d.set_zlabel("Z (cm)")
    ax3d.set_title("A — 3D trajectory (colored by frequency)")
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax3d, label='Frequency (Hz)', shrink=0.6, pad=0.1)

    # --- B: Components vs frequency ---
    ax = fig.add_subplot(gs[0, 1])
    labels_xyz = ['X (cm)', 'Y (cm)', 'Z (cm)']
    colors_xyz = ['C0', 'C1', 'C2']
    for k in range(3):
        ax.plot(f, dr[:, k], alpha=0.25, color=colors_xyz[k], lw=0.8)
        ax.plot(f, ds[:, k], color=colors_xyz[k], lw=1.8,
                label=labels_xyz[k])
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xscale('log')
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Position (cm)")
    ax.set_title("B — Components of d(f)")
    ax.set_ylim(-25,25)
    ax.legend(fontsize=9)

    # --- C: Magnitude |d(f)| ---
    ax = fig.add_subplot(gs[1, 0])
    mag_raw    = np.linalg.norm(dr, axis=1)
    mag_smooth = np.linalg.norm(ds, axis=1)
    ax.plot(f, mag_raw,    alpha=0.25, color='purple', lw=0.8)
    ax.plot(f, mag_smooth, color='purple', lw=2)
    ax.set_ylim(0, 30)
    # Octave band annotations
    for fc, db, ok in zip(OCTAVE_CENTERS, d_bands, valid):
        if ok and f_range[0] <= fc <= f_range[1]:
            ax.axvline(fc, color='gray', lw=0.5, ls='--')
            ax.text(fc, mag_smooth.max() * 0.95,
                    f"{np.linalg.norm(db):.1f}", fontsize=7,
                    ha='center', va='top', color='gray')
    ax.set_xscale('log')
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("|d(f)| (cm)")
    ax.set_title("C — Acoustic center displacement magnitude")
    ax.text(0.98, 0.05, "Numbers = |d| in cm at octave centers",
            transform=ax.transAxes, ha='right', fontsize=7, color='gray')

    # --- D: Fit quality ---
    ax = fig.add_subplot(gs[1, 1])
    ax.semilogy(f, err, color='darkred', lw=1.5)
    ax.set_xscale('log')
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("RMS fit residual (rad)")
    ax.set_title("D — Dipole fit quality (lower = better)")
    ax.text(0.98, 0.95,
            "High residual = non-dipole structure\n(reflections, quadrupole radiation)",
            transform=ax.transAxes, ha='right', va='top', fontsize=7, color='gray')

    if outfile:
        plt.savefig(outfile, bbox_inches='tight')
        print(f"Saved: {outfile}")
    plt.close(fig)


# ---------------------------------------------------------------------------
# 5.  FULL PIPELINE (convenience wrapper)
# ---------------------------------------------------------------------------

def extract_acoustic_center_trajectory(H, freqs, theta, phi_az,
                                        c=343.0, f_range=(200, 8000),
                                        window_oct=0.5, polyorder=3,
                                        violin_id="", outfile=None):
    """
    Complete pipeline: H(f,Omega) → 3D acoustic center trajectory.

    Parameters
    ----------
    H        : complex ndarray (n_freq, n_dirs)
    freqs    : ndarray (n_freq,) Hz
    theta    : colatitude (n_dirs,) rad
    phi_az   : azimuth    (n_dirs,) rad
    c        : speed of sound m/s
    f_range  : (fmin, fmax) for display Hz
    window_oct : SG smoothing window in octaves
    polyorder  : SG polynomial order

    Returns
    -------
    dict with d_raw, d_smooth, fit_error, d_bands
    """
    phi_excess           = compute_excess_phase(H)
    phi_common, phi_res1 = remove_common_phase(phi_excess)

    d_raw, fit_error, phi_dipole = fit_acoustic_center_phase(
        phi_res1, freqs, theta, phi_az, c=c)

    d_smooth = smooth_trajectory(d_raw, freqs, window_oct=window_oct,
                                 polyorder=polyorder)

    d_bands, valid = octave_band_centers(d_smooth, freqs)

    plot_trajectory(d_raw, d_smooth, freqs, fit_error,
                    f_range=f_range, violin_id=violin_id, outfile=outfile)

    return dict(d_raw=d_raw, d_smooth=d_smooth,
                fit_error=fit_error, d_bands=d_bands,
                phi_dipole=phi_dipole, phi_res1=phi_res1)


In [15]:
minfreq = 1000
maxfreq = 10000
for nv in range(6):
    res = extract_acoustic_center_trajectory(
    H[nv,:,:].T, freqs, theta, phi_az,
    c=C, f_range=(minfreq, maxfreq), window_oct=0.5,
    violin_id=f"#{NumViolon[nv]}",
    outfile=f"acoustic_center_trajectory_{nv}.pdf"
    )

    # Print octave summary
    print("\nOctave-band acoustic center positions (smoothed):")
    print(f"{'Freq':>8}  {'X':>8}  {'Y':>8}  {'Z':>8}  {'|d|':>8}  (cm)")
    print("-" * 55)
    for fc, db, ok in zip(OCTAVE_CENTERS, res['d_bands'], [True]*7):
        if minfreq <= fc <= maxfreq and ok:
            print(f"{fc:>8}  {db[0]:>8.2f}  {db[1]:>8.2f}  {db[2]:>8.2f}  "
                    f"{np.linalg.norm(db):>8.2f}")

Saved: acoustic_center_trajectory_0.pdf

Octave-band acoustic center positions (smoothed):
    Freq         X         Y         Z       |d|  (cm)
-------------------------------------------------------
    1000      0.07      1.37     -9.19      9.30
    2000      5.94      5.19     -1.73      8.08
    4000      3.45      5.37     -0.51      6.41
    8000      0.50     -0.05      0.84      0.98
Saved: acoustic_center_trajectory_1.pdf

Octave-band acoustic center positions (smoothed):
    Freq         X         Y         Z       |d|  (cm)
-------------------------------------------------------
    1000     31.80     -0.58    -14.34     34.88
    2000     16.06     -4.20    -10.31     19.54
    4000     10.57     -0.98    -10.48     14.92
    8000      0.18      0.25      0.58      0.66
Saved: acoustic_center_trajectory_2.pdf

Octave-band acoustic center positions (smoothed):
    Freq         X         Y         Z       |d|  (cm)
-------------------------------------------------------
  